In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import time

In [14]:
recipes_clean = pd.read_csv(
    "../data/processed/recipes_clean.csv"
)

print("Recipes dataset loaded!")
print(recipes_clean.shape)

Recipes dataset loaded!
(231637, 15)


In [5]:
import pandas as pd
import numpy as np

interactions_clean = pd.read_csv(
    "../data/processed/interactions_clean.csv"
)

print("Interactions dataset loaded!")
print(interactions_clean.shape)

Interactions dataset loaded!
(1071520, 5)


In [6]:
ratings = interactions_clean[
    ["user_id", "recipe_id", "rating"]
].copy()

print("Total ratings:", len(ratings))
print("Unique users:", ratings["user_id"].nunique())
print("Unique recipes:", ratings["recipe_id"].nunique())

Total ratings: 1071520
Unique users: 196098
Unique recipes: 226590


In [7]:
user_counts = ratings["user_id"].value_counts()
recipe_counts = ratings["recipe_id"].value_counts()

ratings = ratings[
    ratings["user_id"].isin(user_counts[user_counts >= 5].index)
]

ratings = ratings[
    ratings["recipe_id"].isin(recipe_counts[recipe_counts >= 5].index)
]

print("Filtered ratings:", len(ratings))
print("Remaining users:", ratings["user_id"].nunique())
print("Remaining recipes:", ratings["recipe_id"].nunique())

Filtered ratings: 586159
Remaining users: 21973
Remaining recipes: 50716


In [8]:
user_item_matrix = ratings.pivot_table(
    index="user_id",
    columns="recipe_id",
    values="rating",
    fill_value=0
)

print("Matrix shape:", user_item_matrix.shape)

Matrix shape: (21973, 50716)


In [9]:
sparse_matrix = csr_matrix(
    user_item_matrix.values,
    dtype=np.float32
)

print("Sparse matrix shape:", sparse_matrix.shape)
print("Non-zero ratings:", sparse_matrix.nnz)

Sparse matrix shape: (21973, 50716)
Non-zero ratings: 586159


In [10]:
start_time = time.time()

item_similarity = cosine_similarity(
    sparse_matrix.T,
    dense_output=False
)

print("Similarity calculation completed!")
print("Similarity matrix shape:", item_similarity.shape)
print("Time taken:", round(time.time() - start_time, 2), "seconds")

Similarity calculation completed!
Similarity matrix shape: (50716, 50716)
Time taken: 6.13 seconds


In [11]:
item_similarity = item_similarity.tocsr()
print("Similarity matrix is ready!")

Similarity matrix is ready!


In [16]:
def recommend_for_user(user_id, n=10):

    if user_id not in user_item_matrix.index:
        print("User not found!")
        return None

    user_ratings = user_item_matrix.loc[user_id].values

    scores = user_ratings @ item_similarity
    scores = np.asarray(scores).flatten()

    rated_indices = np.where(user_ratings > 0)[0]
    scores[rated_indices] = -1

    top_indices = np.argsort(scores)[::-1][:n]

    recommendations = []

    recipe_ids = user_item_matrix.columns

    for index in top_indices:
        recipe_id = recipe_ids[index]

        recipe = recipes_clean[
            recipes_clean["id"] == recipe_id
        ]

        if not recipe.empty:
            recipe = recipe.iloc[0]

            recommendations.append({
                "Recipe": recipe["name"],
                "Cooking Time": recipe["minutes"],
                "Recommendation Score": round(scores[index], 4)
            })

    return pd.DataFrame(recommendations)

In [15]:
test_user = user_item_matrix.index[0]

print("Testing for User ID:", test_user)

recommend_for_user(test_user, n=10)

Testing for User ID: 1533


,Recipe,Cooking Time,Recommendation Score
0,smothered ground sirloin steak with adobo gravy,40.0,11.2174
1,dukkah,10.0,10.7688
2,balsamic basil carrots,20.0,10.7566
3,easy salmon pasta,30.0,10.0204
4,crunchy seasoned oven fries,45.0,9.9312
5,lime baked fish,28.0,9.6530
6,mustardy cauliflower salad,20.0,9.5924
7,ambrosia oatmeal chocolate chip cookies,10.0,9.5910
8,spicy sesame oven fried potatoes,55.0,9.5286
9,hoisin salmon,11.0,9.5032


In [17]:
def normalize_scores(scores):

    scores = np.array(scores, dtype=float)

    min_score = scores.min()
    max_score = scores.max()

    if max_score == min_score:
        return np.zeros_like(scores)

    return (scores - min_score) / (max_score - min_score)
    

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    min_df=2,
    max_df=0.90,
    dtype=np.float32
)

recipe_vectors = tfidf.fit_transform(recipes_clean["case_text"])

print("Case-Based features loaded!")
print("Matrix shape:", recipe_vectors.shape)

Case-Based features loaded!
Matrix shape: (231637, 37417)


In [19]:
recipe_id_to_row = {
    recipe_id: index
    for index, recipe_id in enumerate(recipes_clean["id"])
}

recipe_lookup = recipes_clean.set_index("id")

print("Recipe lookup created!")

Recipe lookup created!


In [23]:
def recommend_hybrid_for_user(
    user_id, n=10, cf_weight=0.5, case_weight=0.5
):

    if user_id not in user_item_matrix.index:
        print("User not found!")
        return None

    user_ratings = user_item_matrix.loc[user_id].values

    # Find highly rated recipes
    liked_indices = np.where(user_ratings >= 4)[0]

    if len(liked_indices) == 0:
        liked_indices = np.where(user_ratings > 0)[0]

    if len(liked_indices) == 0:
        print("No rating history available!")
        return None

    # Get recipe IDs
    recipe_ids = user_item_matrix.columns.to_numpy()

    liked_recipe_ids = recipe_ids[liked_indices]

    # Find corresponding TF-IDF rows
    liked_rows = [
        recipe_id_to_row[rid]
        for rid in liked_recipe_ids
        if rid in recipe_id_to_row
    ]

    if len(liked_rows) == 0:
        print("No matching recipes found!")
        return None

    # Build user preference profile
    user_profile = np.asarray(
    recipe_vectors[liked_rows].mean(axis=0)
    )
    # Case-Based scores
    case_scores = cosine_similarity(
        user_profile, recipe_vectors
    ).flatten()

    # Collaborative Filtering scores
    cf_scores = np.asarray(
        user_ratings @ item_similarity
    ).flatten()

    # Exclude previously rated recipes
    candidate_indices = np.where(user_ratings == 0)[0]

    candidate_ids = []
    candidate_cf = []
    candidate_case = []

    for index in candidate_indices:

        recipe_id = recipe_ids[index]

        if recipe_id in recipe_id_to_row:

            candidate_ids.append(recipe_id)
            candidate_cf.append(cf_scores[index])

            recipe_row = recipe_id_to_row[recipe_id]
            candidate_case.append(case_scores[recipe_row])

    # Normalize both scores
    cf_normalized = normalize_scores(candidate_cf)
    case_normalized = normalize_scores(candidate_case)

    # Combine scores
    hybrid_scores = (
        cf_weight * cf_normalized +
        case_weight * case_normalized
    )

    # Get top recommendations
    top_indices = np.argsort(hybrid_scores)[::-1][:n]

    recommendations = []

    for index in top_indices:

        recipe_id = candidate_ids[index]
        recipe = recipe_lookup.loc[recipe_id]

        recommendations.append({
            "Recipe": recipe["name"],
            "Cooking Time": recipe["minutes"],
            "Case-Based Score": round(
                case_normalized[index], 4
            ),
            "Collaborative Score": round(
                cf_normalized[index], 4
            ),
            "Hybrid Score": round(
                hybrid_scores[index], 4
            )
        })

    return pd.DataFrame(recommendations)

In [24]:
test_user = 1533

hybrid_results = recommend_hybrid_for_user(
    test_user,
    n=10
)

print("Hybrid Recommendations:")
display(hybrid_results)

Hybrid Recommendations:


,Recipe,Cooking Time,Case-Based Score,Collaborative Score,Hybrid Score
0,potatoes and peas in cream,23.0,0.8531,0.8236,0.8384
1,easy salmon pasta,30.0,0.6537,0.8933,0.7735
2,parmesan potato wedges,245.0,0.7799,0.7178,0.7489
3,mediterranean skillet,65.0,0.7584,0.7258,0.7421
4,braised broccoli with garlic anchovies wine,26.0,0.6459,0.8234,0.7347
5,tomato mushroom pasta,52.0,0.7591,0.7103,0.7347
6,parmesan lemon rice,40.0,0.6643,0.7832,0.7237
7,road kill potatoes,60.0,0.6931,0.7543,0.7237
8,broccoli with creamy lemon sauce,15.0,0.7502,0.6898,0.7200
9,fragrant apricot rice,50.0,0.6317,0.8057,0.7187


In [25]:
import os
import joblib
from sklearn.neighbors import NearestNeighbors

# Create models folder
os.makedirs("../models", exist_ok=True)

print("Preparing models for the application...")

# Train a Case-Based nearest-neighbor model
case_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11,
    n_jobs=-1
)

case_model.fit(recipe_vectors)

# Save all required components
artifacts = {
    "recipes": recipes_clean,
    "user_item_matrix": user_item_matrix,
    "item_similarity": item_similarity,
    "tfidf": tfidf,
    "recipe_vectors": recipe_vectors,
    "case_model": case_model
}

joblib.dump(
    artifacts,
    "../models/recommender_artifacts.joblib",
    compress=3
)

print("Models saved successfully!")
print("File: models/recommender_artifacts.joblib")

Preparing models for the application...
Models saved successfully!
File: models/recommender_artifacts.joblib
